In [ ]:
import pyarrow.parquet as pq

table = pq.read_table("../data/hf_datasets/robot-learning-fs26/data/chunk-000/file-000.parquet")
table

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

df = table.to_pandas()
info = json.loads(Path("../data/hf_datasets/robot-learning-fs26/meta/info.json").read_text())
joint_names = info["features"]["action"]["names"]

episode = df[df["episode_index"] == df.iloc[0]["episode_index"]].sort_values("frame_index").reset_index(drop=True)
s0 = np.asarray(episode.loc[0, "observation.state"], dtype=np.float32)
s1 = np.asarray(episode.loc[1, "observation.state"], dtype=np.float32)
a0 = np.asarray(episode.loc[0, "action"], dtype=np.float32)

print("action close to next state?", np.allclose(a0, s1, atol=1e-3))
print("action close to state delta?", np.allclose(a0, s1 - s0, atol=1e-3))
print("mean |action - state_t|:", np.abs(a0 - s0).mean())
print("mean |state_t+1 - state_t|:", np.abs(s1 - s0).mean())

pd.DataFrame({
    "joint": joint_names,
    "state_t": s0,
    "action_t": a0,
    "state_t_plus_1": s1,
    "state_delta": s1 - s0,
    "action_minus_state": a0 - s0,
})